In [ ]:
Describe la procedencia del dataset (Miner / Hugging Face), indicando que repositories representa repositorios únicos con GH-AW, workflows representa metadatos de archivos .md de workflow, y workflow_bodies almacena el contenido Markdown de cada workflow.

In [1]:
from pathlib import Path
import json
import pandas as pd

RAW_DATA_PATH = Path("data/raw")
PROCESSED_DATA_PATH = Path("data/processed")
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

# Cargar tablas Parquet originales
df_repos = pd.read_parquet(RAW_DATA_PATH / "repositories.parquet")
df_wfs = pd.read_parquet(RAW_DATA_PATH / "workflows.parquet")
df_bodies = pd.read_parquet(RAW_DATA_PATH / "workflow_bodies.parquet")

FileNotFoundError: [Errno 2] No such file or directory: 'data\\raw\\repositories.parquet'

In [ ]:
def summarize_table(df: pd.DataFrame, name: str, pk: str):
    print(f"=== TABLA: {name} ===")
    print(f"Filas: {len(df):,} | Columnas: {df.shape[1]}")
    print(f"Clave Primaria ({pk}) valores únicos: {df[pk].nunique():,}")
    print(df.dtypes)
    print("\n")


summarize_table(df_repos, "repositories", "id")
summarize_table(df_wfs, "workflows", "id")
summarize_table(df_bodies, "workflow_bodies", "workflow_id")

print(f"Repositorios únicos (full_name): {df_repos['full_name'].nunique():,}")
print(
    f"Archivos Markdown únicos (filename por repo): {df_wfs['filename'].nunique():,}"
)

In [ ]:
# 1. Valores nulos por columna
null_report = pd.DataFrame(
    {
        "repositories_nulls": df_repos.isnull().sum(),
        "workflows_nulls": df_wfs.isnull().sum(),
        "bodies_nulls": df_bodies.isnull().sum(),
    }
)

# 2. Claves foráneas huérfanas (Orphan Foreign Keys)
orphaned_wfs = df_wfs[~df_wfs["repository_id"].isin(df_repos["id"])]
orphaned_bodies = df_bodies[~df_bodies["workflow_id"].isin(df_wfs["id"])]

# 3. Filas duplicadas
duplicates_report = {
    "repos_dupes": df_repos.duplicated().sum(),
    "wfs_dupes": df_wfs.duplicated().sum(),
    "bodies_dupes": df_bodies.duplicated().sum(),
}

quality_summary = pd.DataFrame(
    [
        {"Chequeo": "Duplicados exactos", "Resultado": sum(duplicates_report.values())},
        {"Chequeo": "Workflows sin Repositorio (FK huérfana)", "Resultado": len(orphaned_wfs)},
        {"Chequeo": "Bodies sin Workflow (FK huérfana)", "Resultado": len(orphaned_bodies)},
    ]
)
quality_summary

In [ ]:
# Limpieza: Eliminar duplicados si los hubiera y guardar copias purificadas
df_repos_clean = df_repos.drop_duplicates(subset=["id"]).copy()
df_wfs_clean = df_wfs.drop_duplicates(subset=["id"]).copy()
df_bodies_clean = df_bodies.drop_duplicates(subset=["workflow_id"]).copy()

# Deserializar raw_frontmatter_json seguro para el segundo notebook
def safe_json_parse(x):
    try:
        return json.loads(x) if x and isinstance(x, str) else {}
    except Exception:
        return {}

df_wfs_clean["frontmatter_dict"] = df_wfs_clean["raw_frontmatter_json"].apply(safe_json_parse)

# Exportar a eda/data/processed/
df_repos_clean.to_parquet(PROCESSED_DATA_PATH / "repositories.parquet", index=False)
df_wfs_clean.to_parquet(PROCESSED_DATA_PATH / "workflows.parquet", index=False)
df_bodies_clean.to_parquet(PROCESSED_DATA_PATH / "workflow_bodies.parquet", index=False)